# 检测 TPU 型号

依次尝试三种主流框架读取 TPU 设备信息：

1. **JAX** —— `jax.devices()` 直接给出 TPU 设备
2. **PyTorch/XLA** —— `torch_xla` 读取 XLA 设备
3. **TensorFlow** —— `TPUClusterResolver` + `list_logical_devices`

再补充环境变量（`TPU_NAME` / `TPU_ACCELERATOR_TYPE` 等）与 `tpu-info` 命令行作为兜底。

In [ ]:
"""检测 TPU 型号信息（Colab / Google Cloud TPU 环境）。

依次尝试三种主流框架：
1. JAX          —— jax.devices() 直接给出 TPU 设备
2. PyTorch/XLA  —— torch_xla 读取 XLA 设备
3. TensorFlow   —— TPUClusterResolver + list_logical_devices
并以环境变量、系统命令 tpu-info 作为兜底。
"""

import os
import platform
import shutil
import subprocess

print("系统    :", platform.system(), platform.release())
print("Python  :", platform.python_version())
print("CPU 核心:", os.cpu_count())

# ------------------------------------------------------------ 0) 环境变量
print("\n=== 环境变量 ===")
_TPU_ENV_KEYS = (
    "TPU_NAME",
    "TPU_WORKER_ID",
    "TPU_WORKER_HOSTNAMES",
    "TPU_ACCELERATOR_TYPE",      # 例如 v4-8 / v5litepod-4，直接反映型号
    "TPU_CHIPS_PER_HOST_BOUNDS",
    "COLAB_TPU_ADDR",
    "XRT_TPU_CONFIG",
)
_found_env = False
for _key in _TPU_ENV_KEYS:
    _val = os.environ.get(_key)
    if _val:
        print(f"{_key} = {_val}")
        _found_env = True
if not _found_env:
    print("未发现 TPU 相关环境变量")


# ------------------------------------------------------------ 1) JAX
print("\n=== JAX ===")
try:
    import jax

    devices = jax.devices()
    print("全部设备:", devices)
    tpu_devices = [d for d in devices if getattr(d, "platform", "") == "tpu"]
    if tpu_devices:
        print("TPU 数量 :", len(tpu_devices))
        first = tpu_devices[0]
        print("首个设备 :", first)
        kind = getattr(first, "device_kind", None)
        if kind:
            print("设备型号 :", kind)   # 例如 "TPU v4"
        print("本进程可见 TPU:", jax.local_device_count())
    else:
        print("未检测到 JAX TPU（当前平台:",
              getattr(devices[0], "platform", "无") if devices else "无", "）")
except ImportError:
    print("未安装 JAX，跳过（Colab 可 !pip install jax[tpu]）")


# ------------------------------------------------------------ 2) PyTorch/XLA
print("\n=== PyTorch/XLA ===")
try:
    import torch
    try:
        import torch_xla.core.xla_model as xm
        import torch_xla.runtime as xr

        device = xm.xla_device()
        print("XLA 设备  :", device)
        print("设备类型  :", xm.xla_device_hw(device))
        world = (xr.global_device_count()
                 if hasattr(xr, "global_device_count") else xm.xrt_world_size())
        print("全局设备数:", world)
        if torch.cuda.is_available():
            print("注意: 当前为 CUDA 环境，非 TPU")
    except ImportError:
        print("未安装 torch_xla，跳过（Colab 可 !pip install torch_xla）")
except ImportError:
    print("未安装 PyTorch，跳过")


# ------------------------------------------------------------ 3) TensorFlow
print("\n=== TensorFlow ===")
try:
    import tensorflow as tf

    print("TensorFlow:", tf.__version__)
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        print("TPUClusterResolver:", resolver.cluster_spec().as_dict())
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)

        logical = tf.config.list_logical_devices("TPU")
        print("逻辑 TPU 数量:", len(logical))
        for d in logical[:8]:
            print("   ", d.name)

        # 型号：读取 TPU 系统元数据
        try:
            tpu_metadata = resolver.get_tpu_system_metadata()
            print("TPU 型号  :", tpu_metadata.device_type)
            print("TPU 核心数:", tpu_metadata.num_cores_per_replica,
                  "×", tpu_metadata.num_replicas)
        except Exception as _e:
            print("读取 TPU 元数据失败:", _e)
    except Exception as e:
        print("未找到 TPU:", e)
except ImportError:
    print("未安装 TensorFlow，跳过")


# ------------------------------------------------------------ 4) 系统命令兜底
print("\n=== 系统命令 ===")
if shutil.which("tpu-info"):
    try:
        out = subprocess.run(["tpu-info"], capture_output=True, text=True, timeout=20)
        print(out.stdout.strip() or out.stderr.strip())
    except Exception as e:
        print("tpu-info 调用失败:", e)
else:
    print("未找到 tpu-info 命令")

if shutil.which("gcloud"):
    try:
        out = subprocess.run(
            ["gcloud", "compute", "instances", "describe",
             "--format=value(guestAccelerators)"],
            capture_output=True, text=True, timeout=20,
        )
        print("gcloud 加速器信息:", out.stdout.strip() or "(空)")
    except Exception as e:
        print("gcloud 查询失败:", e)